# Finetuning pretrained models on spectrograms
* *https://rumn.medium.com/part-1-ultimate-guide-to-fine-tuning-in-pytorch-pre-trained-model-and-its-configuration-8990194b71e*

* Finetuning - Keep the entire architecture as it is with some minor tweakings at the classifier block [Contains the end linear layers]

In [1]:
# Mount this drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [23]:
import torch
import torchvision.models as models
try:
  from torchinfo import summary
except:
  !pip install -qq torchinfo
finally:
  from torchinfo import summary
import torchvision
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import pathlib
from typing import List, Tuple, Dict
from torchvision import transforms
import os
from torch.utils.data import random_split
from PIL import Image
from torch import nn
from tqdm import tqdm


torch.__version__

'2.6.0+cu124'

In [24]:
# Import VGG16 model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = models.vgg16(weights=models.VGG16_Weights.DEFAULT).to(device) # 13 conv and 3 linear # Weughts -> True -> Load the trained model [ImageNet Dataset]
summary(model)

Layer (type:depth-idx)                   Param #
VGG                                      --
├─Sequential: 1-1                        --
│    └─Conv2d: 2-1                       1,792
│    └─ReLU: 2-2                         --
│    └─Conv2d: 2-3                       36,928
│    └─ReLU: 2-4                         --
│    └─MaxPool2d: 2-5                    --
│    └─Conv2d: 2-6                       73,856
│    └─ReLU: 2-7                         --
│    └─Conv2d: 2-8                       147,584
│    └─ReLU: 2-9                         --
│    └─MaxPool2d: 2-10                   --
│    └─Conv2d: 2-11                      295,168
│    └─ReLU: 2-12                        --
│    └─Conv2d: 2-13                      590,080
│    └─ReLU: 2-14                        --
│    └─Conv2d: 2-15                      590,080
│    └─ReLU: 2-16                        --
│    └─MaxPool2d: 2-17                   --
│    └─Conv2d: 2-18                      1,180,160
│    └─ReLU: 2-19                

In [25]:
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [26]:
Path("/content/drive").exists() # Path works with drive path ✅

True

In [27]:
def find_classes(directory: str) -> Tuple[List[str],Dict[str,int]]:

  """
  Finds class names in a root directory and returns class tuple and class index dictionary
  """
  # Get classes using os.scandir
  classes = sorted(entry.name for entry in os.scandir(directory) if entry.is_dir())

  if not classes:
    raise FileNotFoundError(f"Couldn't find any classes in {directory}")

  # Generate indexes for the class name
  class_to_idx = {class_name:i for i,class_name in enumerate(classes)}
  return classes,class_to_idx


In [28]:
class BirdCallDataset(Dataset):
    """Dataset for experimenting with CNNs on bird call spectrograms."""

    def __init__(self, root_dir: Path, transform: transforms = None, n_explore: int = 2):
        if n_explore <= 0 or n_explore > 5:
            raise ValueError("Invalid n_explore value. Must be between 1 and 5.")

        # Find all classes
        classes, class_to_idx = find_classes(root_dir)  # ignore the original class_to_idx

        # Remove any non-valid directories
        classes = [cls for cls in classes if cls != '.ipynb_checkpoints']

        # Take first n_explore classes
        self.classes = classes[:n_explore]

        # Create class_to_idx starting from 0
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}

        self.paths = []
        self.transform = transform

        for cls in self.classes:
            class_paths = list((pathlib.Path(root_dir) / cls).glob('*.png'))
            self.paths.extend(class_paths)

    def __len__(self):
        """Returns total number of samples"""
        return len(self.paths)

    def load_image(self, index: int) -> Image.Image:
        """Opens an image via path and returns it"""
        image_path = self.paths[index]
        return Image.open(image_path).convert("RGB")  # ensures RGB mode

    def __getitem__(self, index) -> Tuple[torch.Tensor, int, str]:
        """Returns one sample of data and label"""
        img = self.load_image(index)
        class_name = self.paths[index].parent.name
        class_idx = self.class_to_idx[class_name]

        if self.transform:
            img = self.transform(img)

        return img, class_idx

In [29]:
transform = transforms.Compose(
    [
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]) # Image net mean and image net std
    ]
)

root_dir = Path("/content/drive/MyDrive/data")

# data_1_c = BirdCallDataset(root_dir=root_dir, transform=transform, n_explore=1)
# data_2_c = BirdCallDataset(root_dir=root_dir, transform=transform, n_explore=2)
# data_3_c = BirdCallDataset(root_dir=root_dir, transform=transform, n_explore=3)
# data_4_c = BirdCallDataset(root_dir=root_dir, transform=transform, n_explore=4)
data_5_c = BirdCallDataset(root_dir=root_dir, transform=transform, n_explore=5)




In [30]:
# Train and test split

train_size = int(0.8 * len(data_5_c))
test_size = len(data_5_c) - train_size

train_dataset, test_dataset = random_split(data_5_c, [train_size, test_size])

print(f'Train size: {len(train_dataset)}')
print(f'Test size: {len(test_dataset)}')

Train size: 4337
Test size: 1085


In [31]:
# Hyperparameters

BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()
NUM_EPOCHS = 5
LEARNING_RATE = 0.001

In [32]:
train_dataloader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS)
test_dataloader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS)

print(f"Train dataloader: {train_dataloader}, length: {len(train_dataloader)}")
print(f"Test dataloader: {test_dataloader}, length: {len(test_dataloader)}")

Train dataloader: <torch.utils.data.dataloader.DataLoader object at 0x7d467034ae50>, length: 136
Test dataloader: <torch.utils.data.dataloader.DataLoader object at 0x7d4670939a10>, length: 34


In [33]:
136*32

4352

## Finetuning VGG16

In [34]:
summary(model) # No flattening

Layer (type:depth-idx)                   Param #
VGG                                      --
├─Sequential: 1-1                        --
│    └─Conv2d: 2-1                       1,792
│    └─ReLU: 2-2                         --
│    └─Conv2d: 2-3                       36,928
│    └─ReLU: 2-4                         --
│    └─MaxPool2d: 2-5                    --
│    └─Conv2d: 2-6                       73,856
│    └─ReLU: 2-7                         --
│    └─Conv2d: 2-8                       147,584
│    └─ReLU: 2-9                         --
│    └─MaxPool2d: 2-10                   --
│    └─Conv2d: 2-11                      295,168
│    └─ReLU: 2-12                        --
│    └─Conv2d: 2-13                      590,080
│    └─ReLU: 2-14                        --
│    └─Conv2d: 2-15                      590,080
│    └─ReLU: 2-16                        --
│    └─MaxPool2d: 2-17                   --
│    └─Conv2d: 2-18                      1,180,160
│    └─ReLU: 2-19                

In [35]:
last_in_feature = model.classifier[6].in_features

model.classifier[6] = nn.Linear(in_features = last_in_feature,out_features=len(data_5_c.classes))

model.classifier.to(device)
summary(model)

Layer (type:depth-idx)                   Param #
VGG                                      --
├─Sequential: 1-1                        --
│    └─Conv2d: 2-1                       1,792
│    └─ReLU: 2-2                         --
│    └─Conv2d: 2-3                       36,928
│    └─ReLU: 2-4                         --
│    └─MaxPool2d: 2-5                    --
│    └─Conv2d: 2-6                       73,856
│    └─ReLU: 2-7                         --
│    └─Conv2d: 2-8                       147,584
│    └─ReLU: 2-9                         --
│    └─MaxPool2d: 2-10                   --
│    └─Conv2d: 2-11                      295,168
│    └─ReLU: 2-12                        --
│    └─Conv2d: 2-13                      590,080
│    └─ReLU: 2-14                        --
│    └─Conv2d: 2-15                      590,080
│    └─ReLU: 2-16                        --
│    └─MaxPool2d: 2-17                   --
│    └─Conv2d: 2-18                      1,180,160
│    └─ReLU: 2-19                

In [15]:
for x,y in enumerate(range(5,0,-1)):
  print(x,y)

0 5
1 4
2 3
3 2
4 1


In [16]:
a = iter(train_dataloader)
k = next(a)
k[0],k[1]

(tensor([[[[-0.9020, -0.9020, -0.8849,  ..., -0.8849, -0.8849, -0.8849],
           [-0.9363, -0.9363, -0.9363,  ..., -0.9363, -0.9363, -0.9363],
           [-0.9877, -1.0219, -1.0733,  ..., -1.0562, -1.0562, -1.0562],
           ...,
           [-1.0562, -1.0219, -0.9705,  ..., -1.0219, -1.0562, -1.0733],
           [-0.9534, -0.9534, -0.9363,  ..., -0.9705, -1.0390, -1.0562],
           [-0.9192, -0.9192, -0.9192,  ..., -0.9534, -1.0219, -1.0562]],
 
          [[-1.7381, -1.6856, -1.5455,  ..., -1.4755, -1.4230, -1.3880],
           [-1.4930, -1.4230, -1.2829,  ..., -1.2479, -1.2129, -1.1954],
           [-0.9153, -0.8452, -0.6702,  ..., -0.7402, -0.7577, -0.7577],
           ...,
           [-0.7577, -0.8102, -0.9503,  ..., -0.8102, -0.7227, -0.6877],
           [-1.0378, -1.0553, -1.0903,  ..., -0.9503, -0.7752, -0.7052],
           [-1.1604, -1.1604, -1.1429,  ..., -1.0203, -0.7927, -0.7052]],
 
          [[-0.0615, -0.0092,  0.1128,  ...,  0.1651,  0.2173,  0.2522],
           [ 

In [17]:
k[0].shape

torch.Size([32, 3, 224, 224])

In [44]:
a[1] = dict()

In [18]:
# a[1][epoch] = 'ehllo'

In [19]:
#a[1]['']

In [36]:
next(model.parameters()).device

device(type='cuda', index=0)

In [38]:
next(model.classifier.parameters()).device

device(type='cuda', index=0)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = NUM_EPOCHS

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr = LEARNING_RATE, weight_decay=0.01)

results = dict()

for epoch in tqdm(range(NUM_EPOCHS)):

  model.train()

  train_loss = 0
  train_acc = 0
  test_loss = 0
  test_acc = 0


  for batch,(X,y) in enumerate(train_dataloader):

    X = X.to(device)
    y = y.to(device)

    # Forward Pass
    y_pred = model(X)
    loss = criterion(y_pred,y)

    train_loss += loss.item() # Accumulating loss over batches 1 epoch -> loss = loss per corressponding batch

    # Backward Pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    y_pred_class = torch.argmax(torch.softmax(y_pred,dim=1),dim=1)
    train_acc += (y_pred_class == y).sum().item()/len(y_pred)


  # Calulate over an epoch
  train_loss = train_loss / len(train_dataloader)
  train_acc = train_acc / len(train_dataloader)

  results[epoch] = dict()
  results[epoch]['train_loss'] = train_loss
  results[epoch]['train_acc'] = train_acc

  # Testing phase
  model.eval()
  with torch.inference_mode():
    for batch,(X,y) in enumerate(test_dataloader):
      X = X.to(device)
      y = y.to(device)

      # Forward Pass
      y_pred = model(X)
      loss = criterion(y_pred,y)

      # NO BACKWARD HERE

      test_loss += loss.item() # Accumulating loss over batches 1 epoch -> loss = loss per corressponding batch
      y_pred_class = torch.argmax(torch.softmax(y_pred,dim=1),dim=1)
      test_acc += (y_pred_class == y).sum().item()/len(y_pred)

  test_loss = test_loss / len(test_dataloader)
  test_acc = test_acc / len(test_dataloader)

  results[epoch]['test_loss'] = test_loss
  results[epoch]['test_acc'] = test_acc

  0%|          | 0/5 [00:00<?, ?it/s]